In [0]:
# Silver Layer: Payments — Cleansing + CDC MERGE (BATCH ITERATOR)

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, lower, upper, try_to_date, to_timestamp,
    when, coalesce, lit, current_timestamp,
    regexp_replace, row_number, desc
)
from pyspark.sql.types import DoubleType
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

BRONZE_TABLE = "ecomstore.ecomlake.bronze_payments"
SILVER_TABLE = "ecomstore.ecomlake.silver_payments"
QUARANTINE_TABLE = "ecomstore.ecomlake.silver_quarantine_payments"
TEMP_STAGING_TABLE = "ecomstore.ecomlake.temp_payments_staging"

bronze_df = spark.read.table(BRONZE_TABLE)

# 1. Fetch batches chronologically
batches_df = spark.sql(f"SELECT DISTINCT _batch_id FROM {BRONZE_TABLE} ORDER BY _batch_id").collect()
batches = [row['_batch_id'] for row in batches_df]

for current_batch in batches:
    print(f"\n🚀 Processing Batch: {current_batch}")
    
    incoming_batch = bronze_df.filter(col("_batch_id") == current_batch)

    # 2. Deduplication WITHIN the current batch
    w = Window.partitionBy("payment_id").orderBy(desc("updated_at"), desc("_ingestion_timestamp"))
    deduped_df = incoming_batch.withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn")

    # 3. Cleansing & Standardization
    cleaned_df = ( 
        deduped_df
        .withColumn("payment_id", trim(upper(col("payment_id"))))
        .withColumn("order_id",   trim(upper(col("order_id"))))
        .withColumn("payment_method", lower(trim(col("payment_method"))))
        .withColumn("payment_method",
            when(col("payment_method").isin("credit card", "credit_card"), lit("credit_card"))
            .when(col("payment_method").isin("debit card",  "debit_card"),  lit("debit_card"))
            .when(col("payment_method").isin("upi"),                         lit("upi"))
            .when(col("payment_method").isin("net_banking", "net banking"),  lit("net_banking"))
            .when(col("payment_method").isin("cod"),                         lit("cod"))
            .when(col("payment_method").isin("wallet"),                      lit("wallet"))
            .otherwise(lit("other"))
        )
        .withColumn("payment_status", lower(trim(col("payment_status"))))
        .withColumn("payment_status",
            when(col("payment_status").isin("success", "failed", "pending", "refunded"), col("payment_status")).otherwise(lit("unknown"))
        )
        .withColumn("amount", regexp_replace(col("amount").cast("string"), r"[^\d.]", "").cast(DoubleType()))
        .withColumn("amount", when(col("amount") < 0, None).otherwise(col("amount")))
        .withColumn("gateway", when(col("gateway").isNotNull(), lower(trim(col("gateway")))).otherwise(None))
        .withColumn("payment_date", coalesce(try_to_date(col("payment_date"), "yyyy-MM-dd"), try_to_date(col("payment_date"), "dd/MM/yyyy"), try_to_date(col("payment_date"), "MM/dd/yyyy")))
        .withColumn("created_at", coalesce(to_timestamp(col("created_at")), to_timestamp(col("created_at"), "yyyy-MM-dd HH:mm:ss")))
        .withColumn("updated_at", coalesce(to_timestamp(col("updated_at")), to_timestamp(col("updated_at"), "yyyy-MM-dd HH:mm:ss")))
        .withColumn("_silver_processed_at", current_timestamp())
    )

    silver_schema_cols = [
        "payment_id", "order_id", "payment_method", "payment_status",
        "amount", "transaction_id", "payment_date", "gateway",
        "created_at", "updated_at", "_batch_id", "_silver_processed_at"
    ]

    # Disk Staging
    cleaned_df.select(*silver_schema_cols).coalesce(1).write.format("delta").mode("overwrite").saveAsTable(TEMP_STAGING_TABLE)
    staged_silver_df = spark.read.table(TEMP_STAGING_TABLE)

    # 4. Data Quality split
    good_records = staged_silver_df.filter(col("payment_id").isNotNull() & col("order_id").isNotNull() & col("amount").isNotNull() & col("payment_date").isNotNull())
    bad_records = staged_silver_df.filter(col("payment_id").isNull() | col("order_id").isNull() | col("amount").isNull() | col("payment_date").isNull())

    if bad_records.count() > 0:
        (
            bad_records
            .withColumn("rejection_reason",
                when(col("payment_id").isNull(), lit("null_payment_id"))
                .when(col("order_id").isNull(), lit("null_order_id"))
                .when(col("amount").isNull(), lit("null_or_negative_amount"))
                .otherwise(lit("null_payment_date"))
            )
            .coalesce(1)
            .write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(QUARANTINE_TABLE)
        )

    # 5. CDC MERGE
    if spark.catalog.tableExists(SILVER_TABLE):
        silver_target = DeltaTable.forName(spark, SILVER_TABLE)
        (
            silver_target.alias("target")
            .merge(good_records.alias("source"), "target.payment_id = source.payment_id")
            .whenMatchedUpdate(
                condition="source.payment_status != target.payment_status OR source.updated_at > target.updated_at",
                set={c: f"source.{c}" for c in silver_schema_cols if c not in ["payment_id", "created_at"]}
            )
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"✅ CDC MERGE complete for Batch: {current_batch}")
    else:
        good_records.coalesce(1).write.format("delta").mode("overwrite").option("delta.autoOptimize.optimizeWrite", "true").option("delta.autoOptimize.autoCompact", "true").saveAsTable(SILVER_TABLE)

spark.sql(f"DROP TABLE IF EXISTS {TEMP_STAGING_TABLE}")